# Information Extraction with MIPROv2

This notebook uses DSPy's `MIPROv2` optimizer to improve a structured information extraction pipeline for facility support messages.

For each message, the program predicts:

1. **Urgency**: `low`, `medium`, or `high`
2. **Sentiment**: `positive`, `neutral`, or `negative`
3. **Categories**: the relevant facility-management categories

The workflow in this notebook is:
- configure the language model used by DSPy
- load and split the facility-support dataset
- define a multi-part DSPy program for the three extraction subtasks
- evaluate the baseline system
- optimize the prompts with `MIPROv2`
- inspect the optimized instructions and measure the final result

`MIPROv2` is the optimizer actually used in the code cells here, so the markdown below is written specifically for that flow.


In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors AI Lab. <blockquote>This notebook is part of the AI Lab Book companion repository. 
Code is released under the MIT License and accompanying text under CC BY 4.0 - see the LICENSE files at the repository root.

 </blockquote>
 <hr/>
</div>



## Configure Language Model

The language model comes from `LLM_PROVIDER` in `.env`. `supportvectors` uses the SV cluster (`gpt-oss-20b`); any other value uses hosted OpenAI (`gpt-5-mini`).


In [2]:
# LLM_PROVIDER in .env selects the backend:
#   supportvectors -> SV cluster (gpt-oss-20b)
#   anything else  -> hosted OpenAI (gpt-5-mini; needs OPENAI_API_KEY)
import dspy
from svlearn_dspy.llm import make_lm

lm = make_lm()
dspy.configure(lm=lm)


## Load and Prepare Dataset

Fetch the facility support analyzer dataset from the `llama-prompt-ops` repository and convert each record into a `dspy.Example` with `message` as the model input.


In [3]:
import requests
import dspy
import json
import random

def init_dataset():
    # Load from the url
    url = "https://raw.githubusercontent.com/meta-llama/llama-prompt-ops/refs/heads/main/use-cases/facility-support-analyzer/dataset.json"
    dataset = json.loads(requests.get(url).text)
    dspy_dataset = [
        dspy.Example({
            "message": d['fields']['input'],
            "answer": d['answer'],
        }).with_inputs("message")
        for d in dataset
    ]
    random.Random(0).shuffle(dspy_dataset)
    train_set = dspy_dataset[:int(len(dspy_dataset) * 0.33)]
    val_set = dspy_dataset[int(len(dspy_dataset) * 0.33):int(len(dspy_dataset) * 0.66)]
    test_set = dspy_dataset[int(len(dspy_dataset) * 0.66):]

    return train_set, val_set, test_set

## Initialize Dataset Splits

Shuffle the dataset deterministically and divide it into train, validation, and test partitions. The cell prints the split sizes for a quick sanity check.


In [4]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(66, 66, 68)

## Examine Example Data

Display one example message together with its expected answer so we can see the target structure before defining the DSPy program.


In [5]:
print("Input Message:")
print(train_set[0]['message'])

print("\n\nGold Answer:")
for k, v in json.loads(train_set[0]['answer']).items():
    print(f"{k}: {v}")

Input Message:
Subject: Adjusting Bi-Weekly Cleaning Schedule for My Office

Dear ProCare Facility Solutions Support Team,

I hope this message finds you well. My name is Dr. Alex Turner, and I have been utilizing your services for my office space for the past year. I must say, your team's dedication to maintaining a pristine environment has been commendable and greatly appreciated.

I am reaching out to discuss the scheduling of our regular cleaning services. While I find the logistical challenges of coordinating these services intellectually stimulating, I believe we could optimize the current schedule to better suit the needs of my team and our workflow. Specifically, I would like to explore the possibility of adjusting our cleaning schedule to a bi-weekly arrangement, ideally on Tuesdays and Fridays, to ensure our workspace remains consistently clean without disrupting our research activities.

Previously, I have attempted to adjust the schedule through the online portal, but I enc

## Define DSPy Module

Build the information-extraction program used throughout the notebook:
- separate signatures for urgency, sentiment, and categories
- a wrapper module that runs three `ChainOfThought` predictors
- a final structured output assembled from the three predictions


In [6]:
from typing import List, Literal


class FacilitySupportAnalyzerUrgency(dspy.Signature):
    """
    Read the provided message and determine the urgency.
    """
    message: str = dspy.InputField()
    urgency: Literal['low', 'medium', 'high'] = dspy.OutputField()

class FacilitySupportAnalyzerSentiment(dspy.Signature):
    """
    Read the provided message and determine the sentiment.
    """
    message: str = dspy.InputField()
    sentiment: Literal['positive', 'neutral', 'negative'] = dspy.OutputField()

class FacilitySupportAnalyzerCategories(dspy.Signature):
    """
    Read the provided message and determine the set of categories applicable to the message.
    """
    message: str = dspy.InputField()
    categories: List[Literal["emergency_repair_services", "routine_maintenance_requests", "quality_and_safety_concerns", "specialized_cleaning_services", "general_inquiries", "sustainability_and_environmental_practices", "training_and_support_requests", "cleaning_services_scheduling", "customer_feedback_and_complaints", "facility_management_issues"]] = dspy.OutputField()

class FacilitySupportAnalyzerMM(dspy.Module):
    def __init__(self):
        self.urgency_module = dspy.ChainOfThought(FacilitySupportAnalyzerUrgency)
        self.sentiment_module = dspy.ChainOfThought(FacilitySupportAnalyzerSentiment)
        self.categories_module = dspy.ChainOfThought(FacilitySupportAnalyzerCategories)
    
    def forward(self, message: str):
        urgency = self.urgency_module(message=message)
        sentiment = self.sentiment_module(message=message)
        categories = self.categories_module(message=message)

        return dspy.Prediction(
            urgency=urgency.urgency,
            sentiment=sentiment.sentiment,
            categories=categories.categories
        )

program = FacilitySupportAnalyzerMM()

## Define Evaluation Metrics

Create task-level scoring functions and a combined metric that averages the urgency, sentiment, and category scores. `MIPROv2` uses this metric while searching for better prompt instructions.


In [7]:
def score_urgency(gold_urgency, pred_urgency):
    """
    Compute score for the urgency module.
    """
    score = 1.0 if gold_urgency == pred_urgency else 0.0
    return score

def score_sentiment(gold_sentiment, pred_sentiment):
    """
    Compute score for the sentiment module.
    """
    score = 1.0 if gold_sentiment == pred_sentiment else 0.0
    return score

def score_categories(gold_categories, pred_categories):
    """
    Compute score for the categories module.
    Uses the same match/mismatch logic as category accuracy in the score.
    """
    correct = 0
    for k, v in gold_categories.items():
        if v and k in pred_categories:
            correct += 1
        elif not v and k not in pred_categories:
            correct += 1
    score = correct / len(gold_categories)
    return score

def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Computes a score based on agreement between prediction and gold standard for categories, sentiment, and urgency.
    Returns the score (float).
    """
    # Parse gold standard from example
    gold = json.loads(example['answer'])

    # Compute scores for all modules
    score_urgency_val = score_urgency(gold['urgency'], pred.urgency)
    score_sentiment_val = score_sentiment(gold['sentiment'], pred.sentiment)
    score_categories_val = score_categories(gold['categories'], pred.categories)

    # Overall score: average of the three accuracies
    total = (score_urgency_val + score_sentiment_val + score_categories_val) / 3

    return total

## Evaluate Baseline Performance

Evaluate the original program on the test set before optimization so we have a baseline for comparison.


In [8]:
import dspy
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 52.47 / 68 (77.2%): 100%|██████████| 68/68 [00:00<00:00, 75.82it/s]

2026/10/20 16:35:15 INFO dspy.evaluate.evaluate: Average Metric: 52.46666666666667 / 68 (77.2%)


,message,answer,urgency,sentiment,categories,metric
0,"Hey ProCare Support Team, Hope you all are doing great! My name is...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries, tr...",✔️ [0.933]
1,"Hey ProCare Team, Hope you’re all doing well! My name’s Jake, and ...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[routine_maintenance_requests, quality_and_safety_concerns, facili...",✔️ [0.933]
2,"Subject: Assistance Needed for HVAC Maintenance Hi [Receiver], I h...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[routine_maintenance_requests, facility_management_issues]",✔️ [0.633]
3,Subject: A Green Inquiry from a Bill Maher Enthusiast Hey ProCare ...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.967]
4,Subject: Inquiry on Sustainability Practices Dear ProCare Facility...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.633]
...,...,...,...,...,...,...
63,Subject: Inquiry About Your Eco-Friendly Practices Dear ProCare Fa...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries, cu...",✔️ [0.600]
64,Subject: Assistance Needed for Facility Management Issue Dear ProC...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,"[facility_management_issues, quality_and_safety_concerns]",✔️ [0.967]
65,"Subject: Request for Training and Support Hi ProCare Support Team,...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[training_and_support_requests, general_inquiries]",✔️ [0.967]
66,Subject: Concerns About Studio Maintenance and Rent Increase Dear ...,"{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,negative,"[routine_maintenance_requests, facility_management_issues, custome...",✔️ [0.667]


EvaluationResult(score=77.16, results=<list of 68 results>)

## Initialize MIPROv2 Optimizer

Instantiate `MIPROv2` with `auto="light"` (cheap validated budget — swap to `"heavy"` for reference-table-scale gains at much higher token cost) and the notebook's metric. In this setup, `MIPROv2` is responsible for searching for improved instructions for the predictors during compilation.


In [9]:
from dspy import MIPROv2

optimizer = MIPROv2(
    metric=metric,
    # Budget: "light" keeps the validated run cheap; "heavy" gives the
    # reference-table-scale gains at much higher token cost.
    auto="light",
    # gpt-5 family models only accept the default temperature (1.0), so pin
    # the instruction-proposal temperature instead of the 0.5 default.
    init_temperature=1.0,
    num_threads=32,
)

## Run MIPROv2 Optimization

Compile the program with `MIPROv2` on the training split. This is the main optimization step in the notebook and produces a new program with updated predictor instructions.


In [10]:
optimized_program = optimizer.compile(
    program,
    trainset=train_set,
)

2026/10/20 16:35:15 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 31
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 52

2026/10/20 16:35:15 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/10/20 16:35:15 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/10/20 16:35:15 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 29%|██▊       | 4/14 [00:31<01:18,  7.90s/it]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/6


  7%|▋         | 1/14 [00:00<00:00, 17.13it/s]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 5/6


... (85 outputs left out)


Average Metric: 25.63 / 35 (73.2%): 100%|██████████| 35/35 [02:22<00:00,  4.07s/it] 

2026/10/20 17:14:50 INFO dspy.evaluate.evaluate: Average Metric: 25.633333333333333 / 35 (73.2%)
2026/10/20 17:14:50 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 73.24 on minibatch of size 35 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 5', 'Predictor 2: Instruction 2', 'Predictor 2: Few-Shot Set 4'].
2026/10/20 17:14:50 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [67.62, 64.67, 69.05, 67.24, 69.81, 79.24, 77.71, 77.33, 74.29, 77.24, 81.05, 76.38, 67.71, 80.38, 80.0, 76.86, 66.1, 76.0, 67.71, 72.95, 79.14, 76.57, 78.48, 71.14, 75.81, 70.57, 72.38, 79.14, 72.76, 69.43, 73.24]
2026/10/20 17:14:50 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [76.67, 69.36, 78.14, 75.71, 78.14, 79.55, 76.22]
2026/10/20 17:14:50 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 79.55
2026/10/20 17:14:50 INFO dspy.teleprompt.mipro_optimizer_v2: ===============


Average Metric: 39.70 / 52 (76.3%): 100%|██████████| 52/52 [00:05<00:00, 10.22it/s]  

2026/10/20 17:14:55 INFO dspy.evaluate.evaluate: Average Metric: 39.699999999999996 / 52 (76.3%)
2026/10/20 17:14:55 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [76.67, 69.36, 78.14, 75.71, 78.14, 79.55, 76.22, 76.35]
2026/10/20 17:14:55 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 79.55
2026/10/20 17:14:55 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2026/10/20 17:14:55 INFO dspy.teleprompt.mipro_optimizer_v2: 

2026/10/20 17:14:55 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 79.55!


## Inspect Optimized Prompts

Print the instructions learned for each predictor so we can inspect how `MIPROv2` changed the prompts for the three extraction subtasks.


In [11]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: urgency_module.predict
Prompt:
Read the provided message and determine the urgency.
*********************************
Predictor: sentiment_module.predict
Prompt:
You are an expert customer-support sentiment analyst for a facilities services company (ProCare). Read the full customer message and produce two outputs exactly as described below.

1) A chain-of-thought style explanation that documents the step‑by‑step reasoning used to judge the overall tone. Begin this field with the exact prefix:
Reasoning: Let's think step by step in order to
Then give a short, concrete analysis that cites specific cues from the message (examples: explicit praise or thanks, complaint language like "disappointed", "upset", "not satisfied", requests for help vs. cancellation, accusatory phrasing, threats, exclamation/ALL CAPS, repeated negative adjectives, polite neutral requests, informational inquiries, expressions of gratitude). If the message contains mixed or ambiguous signals, explicitly de

## Evaluate Optimized Program

Run the optimized program on the same test evaluation to compare it with the baseline model.


In [12]:
evaluate(optimized_program)

Average Metric: 54.77 / 68 (80.5%): 100%|██████████| 68/68 [03:16<00:00,  2.89s/it]

2026/10/20 17:18:12 INFO dspy.evaluate.evaluate: Average Metric: 54.766666666666666 / 68 (80.5%)


,message,answer,urgency,sentiment,categories,metric
0,"Hey ProCare Support Team, Hope you all are doing great! My name is...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries, tr...",✔️ [0.900]
1,"Hey ProCare Team, Hope you’re all doing well! My name’s Jake, and ...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,"[routine_maintenance_requests, quality_and_safety_concerns, custom...",✔️ [0.933]
2,"Subject: Assistance Needed for HVAC Maintenance Hi [Receiver], I h...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,[routine_maintenance_requests],✔️ [0.667]
3,Subject: A Green Inquiry from a Bill Maher Enthusiast Hey ProCare ...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.967]
4,Subject: Inquiry on Sustainability Practices Dear ProCare Facility...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,neutral,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.967]
...,...,...,...,...,...,...
63,Subject: Inquiry About Your Eco-Friendly Practices Dear ProCare Fa...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,neutral,"[sustainability_and_environmental_practices, general_inquiries, cu...",✔️ [0.933]
64,Subject: Assistance Needed for Facility Management Issue Dear ProC...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,"[facility_management_issues, quality_and_safety_concerns]",✔️ [0.967]
65,"Subject: Request for Training and Support Hi ProCare Support Team,...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,"[training_and_support_requests, customer_feedback_and_complaints]",✔️ [0.967]
66,Subject: Concerns About Studio Maintenance and Rent Increase Dear ...,"{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,negative,"[customer_feedback_and_complaints, routine_maintenance_requests, f...",✔️ [0.633]


EvaluationResult(score=80.54, results=<list of 68 results>)

2026/10/20 17:18:15 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/10/20 17:19:48 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2026/10/20 17:22:13 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
